# Swahili news classification: XLM-RoBERTa fine-tune

Encoder with a classification head rather than a generative model. Trains in a few minutes on a T4 and is compared against the same baseline and test split as the other notebooks.

In [ ]:
%pip install -q -U transformers datasets accelerate scikit-learn
%pip install -q git+https://github.com/MaicyMxtim/habari

In [ ]:
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from habari import evaluate

evaluate.RESULTS_DIR = Path("results")

dataset = load_dataset("masakhane/masakhanews", "swa")
labels = sorted(set(dataset["train"]["category"]))
label_to_id = {name: i for i, name in enumerate(labels)}
id_to_label = {i: name for name, i in label_to_id.items()}

print(f"{len(dataset['train'])} train / {len(dataset['test'])} test, labels: {', '.join(labels)}")
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id_to_label,
    label2id=label_to_id,
).to("cuda")


def full_text(row, limit=1500):
    return (row["headline"] + "\n" + row["text"]).strip()[:limit]


def encode(batch):
    out = tokenizer(
        [f"{h}\n{t}".strip() for h, t in zip(batch["headline"], batch["text"])],
        truncation=True,
        max_length=256,
    )
    out["labels"] = [label_to_id[c] for c in batch["category"]]
    return out


cols = dataset["train"].column_names
train_enc = dataset["train"].map(encode, batched=True, remove_columns=cols)
test_enc = dataset["test"].map(encode, batched=True, remove_columns=cols)
print(train_enc)

In [ ]:
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="xlmr_out",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=20,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_enc,
    data_collator=DataCollatorWithPadding(tokenizer),
)
trainer.train()

In [ ]:
logits = trainer.predict(test_enc).predictions
xlmr_pred = logits.argmax(axis=1)
y_true = np.array([label_to_id[r] for r in dataset["test"]["category"]])

metrics = evaluate.evaluate("xlmr_base_finetuned", y_true, xlmr_pred, labels)
evaluate.print_report(metrics)

In [ ]:
model.save_pretrained("xlmr_swahili_news")
tokenizer.save_pretrained("xlmr_swahili_news")

import shutil

shutil.make_archive("xlmr_swahili_news", "zip", "xlmr_swahili_news")
shutil.make_archive("xlmr_results", "zip", "results")
try:
    from google.colab import files

    files.download("xlmr_swahili_news.zip")
    files.download("xlmr_results.zip")
except Exception as err:
    print("download skipped:", err)